In [38]:
import os
import pandas as pd
import re
import random
import shutil
import pdfplumber
import numpy as np
import time
import ast

In [39]:
df_relatorios_pctcs = pd.read_csv("D:/Backup Tomas/T0mas/Faculdade/IC/Geral/Github CVM/corporate-fraud-in-brazil/data/interim/relatorios_dos_processos/relatorios_pctcs.csv")
df_relatorios_esjs = pd.read_csv("D:/Backup Tomas/T0mas/Faculdade/IC/Geral/Github CVM/corporate-fraud-in-brazil/data/interim/relatorios_dos_processos/relatorios_esjs.csv")

In [40]:
df_relatorios_esjs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1264 entries, 0 to 1263
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         1264 non-null   object
 1   Texto      1264 non-null   object
 2   Escaneado  1264 non-null   bool  
 3   Relatório  1248 non-null   object
dtypes: bool(1), object(3)
memory usage: 31.0+ KB


In [41]:
df_relatorios_pctcs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699 entries, 0 to 698
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         699 non-null    object
 1   Texto      699 non-null    object
 2   Escaneado  699 non-null    bool  
 3   Relatório  681 non-null    object
dtypes: bool(1), object(3)
memory usage: 17.2+ KB


In [42]:
df_relatorios_esjs = df_relatorios_esjs.drop_duplicates()
df_relatórios_pctcs = df_relatorios_pctcs.drop_duplicates()

In [43]:
df_relatorios_esjs.info()
df_relatorios_pctcs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1264 entries, 0 to 1263
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         1264 non-null   object
 1   Texto      1264 non-null   object
 2   Escaneado  1264 non-null   bool  
 3   Relatório  1248 non-null   object
dtypes: bool(1), object(3)
memory usage: 31.0+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699 entries, 0 to 698
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         699 non-null    object
 1   Texto      699 non-null    object
 2   Escaneado  699 non-null    bool  
 3   Relatório  681 non-null    object
dtypes: bool(1), object(3)
memory usage: 17.2+ KB


In [44]:
corpus = df_relatorios_esjs.merge(
    df_relatorios_pctcs,
    how = 'outer',
    on = 'ID')

In [45]:
corpus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1910 entries, 0 to 1909
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           1910 non-null   object
 1   Texto_x      1265 non-null   object
 2   Escaneado_x  1265 non-null   object
 3   Relatório_x  1249 non-null   object
 4   Texto_y      699 non-null    object
 5   Escaneado_y  699 non-null    object
 6   Relatório_y  681 non-null    object
dtypes: object(7)
memory usage: 104.6+ KB


In [46]:
corpus["Texto"] = np.where(
    corpus["Texto_x"].notna() & corpus["Texto_y"].isna(), corpus["Texto_x"],  # só x disponível
    np.where(
        corpus["Texto_x"].isna() & corpus["Texto_y"].notna(), corpus["Texto_y"],  # só y disponível
        np.where(
            (corpus["Escaneado_x"] != corpus["Escaneado_y"]),  # se são diferentes
            np.where(corpus["Escaneado_x"] == "False", corpus["Texto_x"], corpus["Texto_y"]),  
            corpus["Texto_x"]  # se iguais, pega x
        )
    )
)

# --- Relatório ---
corpus["Relatório"] = np.where(
    corpus["Relatório_x"].notna() & corpus["Relatório_y"].isna(), corpus["Relatório_x"],  
    np.where(
        corpus["Relatório_x"].isna() & corpus["Relatório_y"].notna(), corpus["Relatório_y"],  
        np.where(
            (corpus["Escaneado_x"] != corpus["Escaneado_y"]),  
            np.where(corpus["Escaneado_x"] == "False", corpus["Relatório_x"], corpus["Relatório_y"]),  
            corpus["Relatório_x"]  
        )
    )
)


In [47]:
corpus

,ID,Texto_x,Escaneado_x,Relatório_x,Texto_y,Escaneado_y,Relatório_y,Texto,Relatório
0,00783-000022-2016-85,Diário Eletrônico da CVM em 26/06/2019\nCOMISS...,False,RELATÓRIO\nI. OBJETO E ORIGEM\n1. Trata-se de ...,NaN,NaN,NaN,Diário Eletrônico da CVM em 26/06/2019\nCOMISS...,RELATÓRIO\nI. OBJETO E ORIGEM\n1. Trata-se de ...
1,00783-000063-2016-71,Diário Eletrônico da CVM em 31/07/2019\nDOU de...,False,RELATÓRIO\nI. OBJETO\n1. Trata-se de processo ...,NaN,NaN,NaN,Diário Eletrônico da CVM em 31/07/2019\nDOU de...,RELATÓRIO\nI. OBJETO\n1. Trata-se de processo ...
2,00783-000207-2015-17,SEI/CVM - 0597876 - Extrato de Sessão de Julga...,False,RELATÓRIO\nI. OBJETO E ORIGEM\n1. Trata-se de ...,NaN,NaN,NaN,SEI/CVM - 0597876 - Extrato de Sessão de Julga...,RELATÓRIO\nI. OBJETO E ORIGEM\n1. Trata-se de ...
3,00783-000775-2015-18,COMISSÃO DE VALORES MOBILIÁRIOS\nEXTRATO DE SE...,False,RELATÓRIO\nI. OBJETO\n1. Trata-se de processo ...,NaN,NaN,NaN,COMISSÃO DE VALORES MOBILIÁRIOS\nEXTRATO DE SE...,RELATÓRIO\nI. OBJETO\n1. Trata-se de processo ...
4,19957-000073-2024-14,NaN,NaN,NaN,COMISSÃO DE VALORES MOBILIÁRIOS\nPARECER DO CO...,False,DA ORIGEM\n[7]\n2. O presente processo origino...,COMISSÃO DE VALORES MOBILIÁRIOS\nPARECER DO CO...,DA ORIGEM\n[7]\n2. O presente processo origino...
...,...,...,...,...,...,...,...,...,...
1905,SP-2013-00448,COMISSÃO DE VALORES MOBILIÁRIOS\nEXTRATO DE SE...,False,RELATÓRIO\nI OBJETO\n1. Trata-se de Processo A...,NaN,NaN,NaN,COMISSÃO DE VALORES MOBILIÁRIOS\nEXTRATO DE SE...,RELATÓRIO\nI OBJETO\n1. Trata-se de Processo A...
1906,SP-2014-00014,EXTRATO DA SESSÃO DE JULGAMENTO DO PROCESSO AD...,False,R e l a t ó r i o\nI. OBJETO E ORIGEM.\n1. Tra...,NaN,NaN,NaN,EXTRATO DA SESSÃO DE JULGAMENTO DO PROCESSO AD...,R e l a t ó r i o\nI. OBJETO E ORIGEM.\n1. Tra...
1907,SP-2014-00230,COMISSÃO DE VALORES MOBILIÁRIOS\nEXTRATO DE SE...,False,RELATÓRIO\nI – OBJETO E ORIGEM\n1. Trata-se de...,NaN,NaN,NaN,COMISSÃO DE VALORES MOBILIÁRIOS\nEXTRATO DE SE...,RELATÓRIO\nI – OBJETO E ORIGEM\n1. Trata-se de...
1908,SP-2014-00383,EXTRATO DA SESSÃO DE JULGAMENTO DO PROCESSO AD...,False,RELATÓRIO\nI. Do Objeto\n1. Trata-se de Termo ...,NaN,NaN,NaN,EXTRATO DA SESSÃO DE JULGAMENTO DO PROCESSO AD...,RELATÓRIO\nI. Do Objeto\n1. Trata-se de Termo ...


In [48]:
corpus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1910 entries, 0 to 1909
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           1910 non-null   object
 1   Texto_x      1265 non-null   object
 2   Escaneado_x  1265 non-null   object
 3   Relatório_x  1249 non-null   object
 4   Texto_y      699 non-null    object
 5   Escaneado_y  699 non-null    object
 6   Relatório_y  681 non-null    object
 7   Texto        1910 non-null   object
 8   Relatório    1877 non-null   object
dtypes: object(9)
memory usage: 134.4+ KB


In [49]:
# Quantas linhas pegaram o X ou Y por regra do escaneado
print("X prevaleceu (escaneado igual ou ambos X/Y iguais):", 
      ((corpus["Texto"] == corpus["Texto_x"]) & corpus["Texto_x"].notna()).sum())

print("Y prevaleceu (escaneado diferente ou X é NaN):", 
      ((corpus["Texto"] == corpus["Texto_y"]) & corpus["Texto_y"].notna()).sum())

# Quantos NaNs ainda existem
print("NaNs restantes:", corpus["Texto"].isna().sum())

X prevaleceu (escaneado igual ou ambos X/Y iguais): 1262
Y prevaleceu (escaneado diferente ou X é NaN): 648
NaNs restantes: 0


In [52]:
corpus = corpus.drop(columns=[
    "Texto_x", "Escaneado_x", "Relatório_x",
    "Texto_y", "Escaneado_y", "Relatório_y"
])

In [53]:
corpus = corpus.rename(columns={
    'ID': 'id',
    'Texto': 'texto',
    'Relatório': 'relatorio'
})

corpus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1910 entries, 0 to 1909
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         1910 non-null   object
 1   texto      1910 non-null   object
 2   relatorio  1877 non-null   object
dtypes: object(3)
memory usage: 44.9+ KB


In [54]:
corpus.to_csv(r"D:\Backup Tomas\T0mas\Faculdade\IC\Geral\Github CVM\corporate-fraud-in-brazil\data\processed\corpus.csv")